In [1]:
# Imports
import numpy as np
import os
import soundfile as sf 
import librosa
import matplotlib.pyplot as plt
import umap
import pandas as pd

ModuleNotFoundError: No module named 'numpy'

In [2]:
# Function that finds all files with a certain extension in a particular folder
def find_all_files(folders, ext, check_sub_folders=1):
    """Find all files with a .txt extension in all folders in folder list (and subfolders if check_sub_folders = 1 (default))"""
    all_files = []
    if isinstance(folders, str):
        # Make sure input is a list of folders
        folders = [folders]
    for f, folder in enumerate(folders):
        for subfolder in os.listdir(folder):

            if os.path.isdir(os.path.join(folder, subfolder)):
                if check_sub_folders:
                    folders.append(os.path.join(folder, subfolder))                
            elif subfolder.endswith(ext):
                all_files.append(os.path.join(folder,subfolder))
    return all_files

In [3]:
# Function that pads or cuts a spectrogram to a certain size, adapted from avgn
def pad_spectrogram(spectrogram, pad_length):
    """ Pads a spectrogram to being a certain length
    """
    excess_needed = pad_length - np.shape(spectrogram)[1]
    pad_left = np.floor(float(excess_needed) / 2).astype("int")
    pad_right = np.ceil(float(excess_needed) / 2).astype("int")
    if excess_needed >= 0:
        return np.pad(
            spectrogram, [(0, 0), (pad_left, pad_right)], "constant", constant_values=0
        )
    else:
        pad_left = np.floor(float(excess_needed) / 2).astype("int")
        pad_right = np.ceil(float(excess_needed) / 2).astype("int")
        return spectrogram[:,-1*pad_left:pad_right]

## Important variables
There are a few variables that will influence the outcome of especially spectral features. These are:
- fmin and fmax - the minimum and maximum frequency to consider when calculating variables. Usually used to cut off low-frequency and high-frequency noise. This needs to be adapted to the species, these values need to be higher when targeting USVs than audible vocalizations for example.
- spec_shape - this is the shape of the spectrogram for the latent feature analysis. You want this to be a multiple of 2. The optimal size depends on the samplerate of the recordings and the average length of the vocalizations
- nfft - when calculating a spectrogram, you typically take samples of nfft length and calculate the frequency spectrum in this smaller sample. You then slide this window across your recordings to get the frequencies at each time point. Ideally n_fft is a power of 2 (for faster computation), the optimum again depends on the samplerate and length of the vocalizations
- hop_length is the number of samples you slide the n_fft window. A smaller number results in a higher time resolution, but more smoothed result. 

In [4]:
# Constant variables
fmin = 1000
fmax = 10000 # Max frequency cutoff
spec_shape = [128, 128] #Shape of spectrogram
n_fft = 2048
hop_length = int(n_fft/2)


In [5]:
# Find all the audio files
data_folder = r"C:\Users\grijseels\Documents\Github\AnimalBehaviourLiveWorkshop\test_data"
data_files = find_all_files(data_folder, '.flac')

NameError: name 'os' is not defined

## Example call
We will first look at some features on a single audio fragment, a marmoset phee call. First, we need to load the call. Then we will plot the spectrogram using librosa to have an idea of what it looks like. This is also a good point to check if our n_fft, hop_length, and frequency settings seem reasonable.

Next we'll calculate the fundamental frequency and the peak frequency. We'll look at how good it is at estimating this from the given audio fragment.

In [6]:
## Example data file
file = data_files[200]
data, sr = sf.read(file) # read in the file using soundfile

NameError: name 'data_files' is not defined

In [7]:
# Calculate a mel spectrogram of the data for better visualization
spec = librosa.feature.melspectrogram(y=data, sr=sr, n_mels=128,fmax=10000, n_fft=nfft, hop_length = hop_length)

# Plot the spectogram in decibels (a log-scale)
fig, ax = plt.subplots()
S_dB = librosa.power_to_db(spec, ref=np.max)
img = librosa.display.specshow(S_dB, x_axis='time',
                         y_axis='mel', sr=sr,
                         fmax=10000, ax=ax)
fig.colorbar(img, ax=ax, format='%+2.0f dB')

NameError: name 'librosa' is not defined

In [8]:
# Calculate the fundamental frequency and voiced (i.e. tonal) portion
f0, voiced_flag, voiced_probs = librosa.pyin(data, sr=sr, frame_length = nfft, hop_length = hop_length,fmin=fmin, fmax=fmax)
   
# Calculate the peak frequency
spec =  np.abs(librosa.stft(y=data, n_fft = nfft, hop_length = hop_length))
freqs = librosa.fft_frequencies(sr=sr, n_fft=nfft)
spec_bp = spec[np.logical_and(freqs>1000, freqs<10000), :]
freqs_bp= freqs [np.logical_and(freqs>1000, freqs<10000)]
peak_freq = freqs_bp[np.argmax(spec_bp, 0)]

NameError: name 'librosa' is not defined

In [ ]:
# Visualization
fig, ax = plt.subplots()
img = librosa.display.specshow(np.log(spec), y_axis='log', x_axis='time', sr = sr, ax=ax, n_fft = nfft, hop_length = hop_length, fmin=fmin, fmax=fmax)
times = librosa.times_like(spec, sr=sr, n_fft = nfft, hop_length = hop_length)
ax.plot(times, peak_freq)
ax.plot(times, f0, 'r')
